# 260513 Adaptive RAG 구현 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w9_agent_rag/llm_260513_adaptive_rag_2.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 강의 메모: Adaptive RAG 그래프 구성 (classify → direct/rag/web/multihop)

- **왜 Adaptive인가**: 일반 RAG는 **모든 쿼리마다 벡터 스토어를 검색**해서 비용이 비싸다. "파이썬 리스트가 뭐야?" 같은 상식 질문에도 청크를 가져오는 게 문제. → LLM이 라우팅을 판단해서 필요한 경로만 타게 한다.
- **AdaptiveState 누적 패턴**: `documents`/`web_results`/`sub_queries`/`retrieved`는 `Annotated[list[str], add]`로 **append 누적**. `hop_count`/`max_hops`로 멀티홉 루프 제어.
- **classify_node = 어제의 router**: `llm.with_structured_output(_Route)` + Pydantic `Literal["direct","rag","web","multihop"]` → 분류 결과가 곧 라우팅 키와 1:1.
- **노드 = LangChain을 그대로 감싼 함수**: rag_node 안에서 `retriever.invoke()` → context join → `llm.invoke()`. LangGraph라고 특별한 게 아니라 **입출력만 `AdaptiveState`/dict로 맞추면 끝**.
- **fan-in / fan-out 패턴**: direct/rag/web이 각자 END로 가지 않고 **공통 `finalize_node`로 모인 뒤** END로. 타임스탬프·출처 라벨·참고문서 건수 같은 **공통 후처리** 일원화. (비유: 갈라졌다가 다시 합쳐지는 강줄기)
- **LangChain vs LangGraph 갈림길**: 단방향이면 LCEL이 직관적. **루프·순환·외부 툴 끼워넣기**가 필요해지는 순간 LangGraph 승. LangChain으로 흉내내려면 `RunnableLambda`를 복잡하게 엮어야 함.
- **라이브 디버깅 함정**: 같은 쿼리에 갑자기 다른 경로가 잡혔던 사건 → 노드 add/엣지 재연결 중 **이전 상태가 꼬여 retrieved에 잔여물**. 그래프 수정할 때마다 컴파일 새로, 상태 초기화 확인.

## 강의 메모: 실무 팁 (멀티홉 루프 · 방어 로직 · GraphRAG)

- **멀티홉 = IRCoT (Interleaving Retrieval CoT)**: 단방향 CoT와 달리 **think ↔ retrieve를 순환**시켜 sub-query를 한 단계씩 만들고 결과 누적. 알렌AI 2022(arxiv 2212.10509). `_Think(next_sub_query, can_answer)`로 LLM이 직접 종료 시점 결정.
- **무한 루프 방지 3중 안전장치**: ① `hop_count >= max_hops` 강제 종료 ② **중복 sub_query** 들어오면 `can_answer=True`로 finalize 직행 ③ 컴파일 시 **전체 노드 실행 횟수 50~100** 캡. 토큰 무한 소비 방지가 핵심.
- **라이브 버그**: `f"[hop{hop}] r.next_sub_query"`처럼 **중괄호 빼먹으면 문자열 그대로 박힘** → `{r.next_sub_query}`로 보간 확인.
- **classify 실패 fallback**: `safe_classify_node`로 try/except 감싸고 실패 시 **`direct`로 폴백** + `CLASSIFY_COUNTER`로 실패 추적. 웹서치 등 외부 API도 동일하게 fail-safe edge 필수.
- **GraphRAG ≠ LangGraph 루프**: GraphRAG는 **DB 자체를 엔티티-관계 그래프로 인덱싱**, LangGraph 루프는 **검색 흐름만 그래프화**. sub-query로 나눠도 코사인 유사도 청크의 본질적 한계는 안 사라짐.
- **트렌드 주의**: 올초엔 GraphRAG가 "안 쓰면 망함" 분위기였지만 지금은 **PageIndex**(벡터스토어를 책처럼 목차/페이지로 인덱싱)가 PDF에서 더 잘 나오기도. 한 기법 올인 금지, 데이터에 맞춰 비교 검증.